# Prepare the crop health dataset (Colab)

Downloads the cassava, maize and beans datasets from Harvard Dataverse (~95 GB of archives), keeps the images at 512 px, builds `annotations.csv`, and saves everything as **one zip in Google Drive** for the team.

Run this **once**. Afterwards everyone loads the zip with the same command (see the last section).

**Before you start**
- Runtime > Change runtime type > **CPU**. No GPU is needed, so this doesn't use GPU quota.
- Expected time: about 1 to 2 hours. Keep this tab open; Colab can disconnect idle sessions.
- Drive needs ~20 GB free while it runs (finished datasets are saved one by one, then combined into a ~10 GB zip and the pieces are deleted).
- If the session disconnects, run all cells again. Each finished dataset is saved to Drive (`crop-data-parts/`), so a rerun restores those and only downloads what is left.
- Keep your Mac awake while it runs: `caffeinate -i -s` in Terminal.

## 1. Settings

In [ ]:
# Where the shared dataset zip is stored in Google Drive
DRIVE_ZIP = "/content/drive/MyDrive/crop-health/crop-data.zip"

REPO_URL = "https://github.com/Gabin-ishimwe/crop-health-model.git"
BRANCH = "feat/per-crop-training"  # change to "main" once the PR is merged
REPO_DIR = "/content/crop-health-model"

## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    size_gb = os.path.getsize(DRIVE_ZIP) / 1024**3
    print(f"The dataset zip already exists ({size_gb:.1f} GB): {DRIVE_ZIP}")
    print("Nothing to prepare. Delete it from Drive first if you really want to rebuild it.")
else:
    print(f"No zip yet. It will be created at {DRIVE_ZIP}")

## 3. Get the code and install what data preparation needs

In [ ]:
# DEBIAN_FRONTEND=noninteractive stops apt from pausing to ask setup questions.
# unar extracts the .rar archives; rarfile is the Python wrapper around it.
# Colab already has torch, torchvision, pandas, pillow, httpx and tqdm, so the
# project itself is not pip-installed (that would replace Colab's torch).
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y unar > /dev/null
!pip install -q rarfile

if not os.path.exists(REPO_DIR):
    # GIT_TERMINAL_PROMPT=0: fail with an error instead of asking for a login
    !GIT_TERMINAL_PROMPT=0 git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 4. Check disk space

The largest dataset needs about 40 GB of local disk while it is being extracted.

In [ ]:
import shutil

free_gb = shutil.disk_usage("/content").free / 1024**3
print(f"Free local disk: {free_gb:.0f} GB")
if free_gb < 50:
    print("Warning: less than 50 GB free, the largest dataset may not fit.")

## 5. Download, prepare and upload

This is the long step. It works through the 7 datasets one at a time: download, extract, resize, delete the archives, and save the finished dataset to Drive. Then it builds `annotations.csv`, writes the full zip to Drive and removes the per-dataset pieces.

If a dataset fails (`Failed to extract`, `Giving up on`), it is not saved and the full zip is not created. Run this cell again: finished datasets are restored from Drive and only the failed one is retried.

In [ ]:
!python -m crop_health_model.data.make_dataset --drive-zip "{DRIVE_ZIP}"

## 6. Check the result

In [ ]:
import json
import zipfile

import pandas as pd

df = pd.read_csv(".data/annotations.csv")
print(f"{len(df)} images in annotations.csv\n")
print(df.groupby(["crop_type", "label"]).size().rename("images").to_string())
print()
print(f"Duplicate image paths: {df['image'].duplicated().sum()}")

with zipfile.ZipFile(DRIVE_ZIP) as zf:
    manifest = json.loads(zf.read("manifest.json"))
print(f"\nZip on Drive: {os.path.getsize(DRIVE_ZIP) / 1024**3:.1f} GB, manifest: {manifest}")

Compare the counts with the expected numbers printed during step 5 (`Expected ... images for class ..., found ...`). Small differences are normal, since unreadable images are skipped.

## 7. Finish uploading to Drive

Run this before closing the tab, so the zip is fully written to Drive.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. The dataset zip is ready for the team.")

## Using the dataset afterwards

In any later session (Colab or a Mac with Google Drive for desktop), from the repo root:

```
python -m crop_health_model.data.make_dataset --drive-zip "/content/drive/MyDrive/crop-health/crop-data.zip"
```

Because the zip exists, this only copies and extracts it into `.data`, which takes a few minutes. Share the `crop-health` Drive folder with the team so everyone can use it.